# TinyViT Training Notebook

This notebook allows you to run TinyViT experiments on Google Colab.

**Supported experiments:**
- Standard training (from scratch or pretrained)
- Knowledge distillation with saved logits
- Online feature + logits distillation

**Instructions:**
1. Run the Setup cell to install dependencies
2. Configure your experiment in the Configuration cell
3. Run the Training cell

## 1. Setup

In [2]:
#@title Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#@title Clone Repository & Install Dependencies
import os

# Clone the repo (modify URL if using your own fork)
REPO_URL = "https://github.com/ilyass7m/TinyViT.git"  #@param {type:"string"}
BRANCH = "ilyas"  #@param {type:"string"}

if not os.path.exists('/content/TinyViT'):
    !git clone -b {BRANCH} {REPO_URL} /content/TinyViT
else:
    print("Repository already cloned")
    !cd /content/TinyViT && git pull

# Change to working directory
%cd /content/TinyViT/Cream/TinyViT

# Install dependencies
!pip install -q timm yacs wandb

Cloning into '/content/TinyViT'...
remote: Enumerating objects: 478, done.
remote: Counting objects: 100% (478/478), done.
remote: Compressing objects: 100% (306/306), done.
remote: Total 478 (delta 176), reused 435 (delta 133), pack-reused 0 (from 0)
Receiving objects: 100% (478/478), 5.12 MiB | 14.17 MiB/s, done.
Resolving deltas: 100% (176/176), done.
[Errno 2] No such file or directory: '/content/TinyViT/Cream/TinyViT'
/content


In [4]:
#@title Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cpu
CUDA available: False


In [ ]:
#@title Download CIFAR-100 Dataset
import torchvision
torchvision.datasets.CIFAR100(root='./data', train=True, download=True)
torchvision.datasets.CIFAR100(root='./data', train=False, download=True)
print("CIFAR-100 downloaded successfully!")

In [ ]:
#@title Download Pretrained Checkpoints (Optional)
import os

os.makedirs('pretrained', exist_ok=True)

# TinyViT-21M pretrained on ImageNet-22K with distillation
DOWNLOAD_21M = True  #@param {type:"boolean"}
if DOWNLOAD_21M and not os.path.exists('pretrained/tiny_vit_21m_22k_distill.pth'):
    !wget -q -P pretrained/ https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_21m_22k_distill.pth
    print("Downloaded TinyViT-21M checkpoint")

# TinyViT-5M pretrained on ImageNet-22K with distillation
DOWNLOAD_5M = True  #@param {type:"boolean"}
if DOWNLOAD_5M and not os.path.exists('pretrained/tiny_vit_5m_22k_distill.pth'):
    !wget -q -P pretrained/ https://github.com/wkcn/TinyViT-model-zoo/releases/download/checkpoints/tiny_vit_5m_22k_distill.pth
    print("Downloaded TinyViT-5M checkpoint")

!ls -la pretrained/

## 2. Configuration

Configure your experiment below. All parameters can be modified.

In [ ]:
#@title Experiment Configuration

# ============================================================================
# EXPERIMENT SELECTION
# ============================================================================
EXPERIMENT_TYPE = "finetune_teacher"  #@param ["scratch", "finetune_teacher", "distill_saved_logits", "online_distill", "online_distill_features"]

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
MODEL_NAME = "TinyViT-21M-CIFAR100"  #@param {type:"string"}

# Student model (TinyViT-5M or TinyViT-21M)
STUDENT_TYPE = "tiny_vit_5m"  #@param ["tiny_vit_5m", "tiny_vit_21m"]

# Model architectures
MODEL_CONFIGS = {
    "tiny_vit_5m": {
        "EMBED_DIMS": [64, 128, 160, 320],
        "DEPTHS": [2, 2, 6, 2],
        "NUM_HEADS": [2, 4, 5, 10],
        "WINDOW_SIZES": [7, 7, 14, 7],
        "FEATURE_DIM": 320,
    },
    "tiny_vit_21m": {
        "EMBED_DIMS": [96, 192, 384, 576],
        "DEPTHS": [2, 2, 6, 2],
        "NUM_HEADS": [3, 6, 12, 18],
        "WINDOW_SIZES": [7, 7, 14, 7],
        "FEATURE_DIM": 576,
    },
}

STUDENT_CONFIG = MODEL_CONFIGS[STUDENT_TYPE]

# ============================================================================
# DATA CONFIGURATION
# ============================================================================
DATASET = "cifar100"  #@param ["cifar100", "imagenet"]
NUM_CLASSES = 100 if DATASET == "cifar100" else 1000
IMG_SIZE = 224  #@param {type:"integer"}
BATCH_SIZE = 64  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}
DATA_PATH = "./data"  #@param {type:"string"}

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================
EPOCHS = 50  #@param {type:"integer"}
WARMUP_EPOCHS = 5  #@param {type:"integer"}
BASE_LR = 2e-4  #@param {type:"number"}
MIN_LR = 1e-6  #@param {type:"number"}
WEIGHT_DECAY = 0.05  #@param {type:"number"}
CLIP_GRAD = 5.0  #@param {type:"number"}
LAYER_LR_DECAY = 0.75  #@param {type:"number"}

# ============================================================================
# PRETRAINED WEIGHTS
# ============================================================================
# For finetuning or as teacher checkpoint
PRETRAINED_PATH = "pretrained/tiny_vit_21m_22k_distill.pth"  #@param {type:"string"}

# ============================================================================
# DISTILLATION CONFIGURATION (for distill experiments)
# ============================================================================
# Teacher model type (for online distillation)
TEACHER_TYPE = "tiny_vit_21m"  #@param ["tiny_vit_5m", "tiny_vit_21m", "vit_base_patch16_224"]
TEACHER_CHECKPOINT = "pretrained/tiny_vit_21m_22k_distill.pth"  #@param {type:"string"}

# Saved logits path (for saved logits distillation)
TEACHER_LOGITS_PATH = "./output/logits/tinyvit21m/"  #@param {type:"string"}
LOGITS_TOPK = 100  #@param {type:"integer"}

# Feature distillation
FEATURE_ENABLED = True  #@param {type:"boolean"}
FEATURE_WEIGHT = 0.5  #@param {type:"number"}

# Temperature for KL divergence
TEMPERATURE = 1.0  #@param {type:"number"}

# ============================================================================
# AUGMENTATION
# ============================================================================
COLOR_JITTER = 0.4  #@param {type:"number"}
AUTO_AUGMENT = "rand-m9-mstd0.5-inc1"  #@param {type:"string"}
MIXUP = 0.0  #@param {type:"number"}
CUTMIX = 0.0  #@param {type:"number"}

# ============================================================================
# OUTPUT CONFIGURATION
# ============================================================================
OUTPUT_DIR = "./output/experiment"  #@param {type:"string"}
SAVE_FREQ = 10  #@param {type:"integer"}
PRINT_FREQ = 50  #@param {type:"integer"}

# Weights & Biases logging
USE_WANDB = False  #@param {type:"boolean"}
WANDB_PROJECT = "TinyViT-CIFAR100"  #@param {type:"string"}

# ============================================================================
# SEED
# ============================================================================
SEED = 42  #@param {type:"integer"}

print(f"Experiment type: {EXPERIMENT_TYPE}")
print(f"Student model: {STUDENT_TYPE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

In [ ]:
#@title Generate Config YAML
import yaml
import os

# Determine teacher config if needed
TEACHER_CONFIG = MODEL_CONFIGS.get(TEACHER_TYPE, MODEL_CONFIGS["tiny_vit_21m"])

# Build config dictionary
config = {
    "MODEL": {
        "NAME": MODEL_NAME,
        "TYPE": "tiny_vit",
        "NUM_CLASSES": NUM_CLASSES,
        "DROP_PATH_RATE": 0.0,
        "DROP_RATE": 0.0,
        "LABEL_SMOOTHING": 0.0,
        "TINY_VIT": {
            "EMBED_DIMS": STUDENT_CONFIG["EMBED_DIMS"],
            "DEPTHS": STUDENT_CONFIG["DEPTHS"],
            "NUM_HEADS": STUDENT_CONFIG["NUM_HEADS"],
            "WINDOW_SIZES": STUDENT_CONFIG["WINDOW_SIZES"],
        },
    },
    "DATA": {
        "DATASET": DATASET,
        "IMG_SIZE": IMG_SIZE,
        "INTERPOLATION": "bicubic",
        "BATCH_SIZE": BATCH_SIZE,
        "NUM_WORKERS": NUM_WORKERS,
        "PIN_MEMORY": True,
    },
    "TRAIN": {
        "EPOCHS": EPOCHS,
        "WARMUP_EPOCHS": WARMUP_EPOCHS,
        "BASE_LR": BASE_LR,
        "WARMUP_LR": 1e-7,
        "MIN_LR": MIN_LR,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "CLIP_GRAD": CLIP_GRAD,
        "LAYER_LR_DECAY": LAYER_LR_DECAY,
        "AUTO_RESUME": True,
    },
    "DISTILL": {
        "ENABLED": False,
        "ONLINE_DISTILL": False,
        "FEATURE_ENABLED": False,
    },
    "AUG": {
        "COLOR_JITTER": COLOR_JITTER,
        "AUTO_AUGMENT": AUTO_AUGMENT,
        "REPROB": 0.25,
        "REMODE": "pixel",
        "RECOUNT": 1,
        "MIXUP": MIXUP,
        "CUTMIX": CUTMIX,
    },
    "TEST": {
        "CROP": True,
    },
    "SEED": SEED,
    "PRINT_FREQ": PRINT_FREQ,
    "SAVE_FREQ": SAVE_FREQ,
}

# Configure based on experiment type
if EXPERIMENT_TYPE == "distill_saved_logits":
    config["DISTILL"]["ENABLED"] = True
    config["DISTILL"]["TEACHER_LOGITS_PATH"] = TEACHER_LOGITS_PATH
    config["DISTILL"]["LOGITS_TOPK"] = LOGITS_TOPK
    config["AUG"]["MIXUP"] = 0.0
    config["AUG"]["CUTMIX"] = 0.0

elif EXPERIMENT_TYPE == "online_distill":
    config["DISTILL"]["ONLINE_DISTILL"] = True
    config["DISTILL"]["TEACHER_CHECKPOINT"] = TEACHER_CHECKPOINT
    config["DISTILL"]["TEACHER_TYPE"] = "tiny_vit" if "tiny_vit" in TEACHER_TYPE else TEACHER_TYPE
    config["DISTILL"]["TEMPERATURE"] = TEMPERATURE
    config["DISTILL"]["FEATURE_ENABLED"] = False
    if "tiny_vit" in TEACHER_TYPE:
        config["DISTILL"]["TEACHER_TINY_VIT"] = {
            "EMBED_DIMS": TEACHER_CONFIG["EMBED_DIMS"],
            "DEPTHS": TEACHER_CONFIG["DEPTHS"],
            "NUM_HEADS": TEACHER_CONFIG["NUM_HEADS"],
            "WINDOW_SIZES": TEACHER_CONFIG["WINDOW_SIZES"],
        }
    config["AUG"]["MIXUP"] = 0.0
    config["AUG"]["CUTMIX"] = 0.0

elif EXPERIMENT_TYPE == "online_distill_features":
    config["DISTILL"]["ONLINE_DISTILL"] = True
    config["DISTILL"]["TEACHER_CHECKPOINT"] = TEACHER_CHECKPOINT
    config["DISTILL"]["TEACHER_TYPE"] = "tiny_vit" if "tiny_vit" in TEACHER_TYPE else TEACHER_TYPE
    config["DISTILL"]["TEMPERATURE"] = TEMPERATURE
    config["DISTILL"]["FEATURE_ENABLED"] = FEATURE_ENABLED
    config["DISTILL"]["FEATURE_WEIGHT"] = FEATURE_WEIGHT
    config["DISTILL"]["FEATURE_DIM_STUDENT"] = STUDENT_CONFIG["FEATURE_DIM"]
    config["DISTILL"]["FEATURE_DIM_TEACHER"] = TEACHER_CONFIG["FEATURE_DIM"]
    if "tiny_vit" in TEACHER_TYPE:
        config["DISTILL"]["TEACHER_TINY_VIT"] = {
            "EMBED_DIMS": TEACHER_CONFIG["EMBED_DIMS"],
            "DEPTHS": TEACHER_CONFIG["DEPTHS"],
            "NUM_HEADS": TEACHER_CONFIG["NUM_HEADS"],
            "WINDOW_SIZES": TEACHER_CONFIG["WINDOW_SIZES"],
        }
    config["AUG"]["MIXUP"] = 0.0
    config["AUG"]["CUTMIX"] = 0.0

# Save config to file
os.makedirs('configs/colab', exist_ok=True)
config_path = 'configs/colab/experiment.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to {config_path}")
print("\n" + "="*50)
print("Generated Configuration:")
print("="*50)
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

## 3. Training

In [ ]:
#@title Build Training Command

# Base command
cmd_parts = [
    "python main.py",
    f"--cfg {config_path}",
    f"--data-path {DATA_PATH}",
    f"--output {OUTPUT_DIR}",
]

# Add pretrained weights for finetuning
if EXPERIMENT_TYPE in ["finetune_teacher", "scratch"] and PRETRAINED_PATH:
    if EXPERIMENT_TYPE == "finetune_teacher":
        cmd_parts.append(f"--pretrained {PRETRAINED_PATH}")

# Add wandb
if USE_WANDB:
    cmd_parts.append("--use-wandb")
    cmd_parts.append(f"--wandb-run-name {MODEL_NAME}")

# Set environment variables for single-GPU training
env_vars = "RANK=0 WORLD_SIZE=1 LOCAL_RANK=0 MASTER_ADDR=localhost MASTER_PORT=29500"

# Full command
full_cmd = f"{env_vars} {' '.join(cmd_parts)}"

print("Training command:")
print("="*50)
print(full_cmd)
print("="*50)

In [ ]:
#@title Run Training
import os

# Set environment variables
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'
os.environ['LOCAL_RANK'] = '0'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '29500'

# Build command without env vars (they're set above)
cmd_parts_clean = [
    "python main.py",
    f"--cfg {config_path}",
    f"--data-path {DATA_PATH}",
    f"--output {OUTPUT_DIR}",
]

if EXPERIMENT_TYPE == "finetune_teacher" and PRETRAINED_PATH:
    cmd_parts_clean.append(f"--pretrained {PRETRAINED_PATH}")

if USE_WANDB:
    cmd_parts_clean.append("--use-wandb")
    cmd_parts_clean.append(f"--wandb-run-name {MODEL_NAME}")

train_cmd = ' '.join(cmd_parts_clean)
print(f"Running: {train_cmd}")
print("="*50)

!{train_cmd}

## 4. Evaluation

In [ ]:
#@title Evaluate Best Checkpoint
import glob

# Find best checkpoint
checkpoint_pattern = f"{OUTPUT_DIR}/**/ckpt_best.pth"
checkpoints = glob.glob(checkpoint_pattern, recursive=True)

if checkpoints:
    best_ckpt = checkpoints[0]
    print(f"Found checkpoint: {best_ckpt}")
    
    eval_cmd = f"python main.py --cfg {config_path} --data-path {DATA_PATH} --output {OUTPUT_DIR} --resume {best_ckpt} --eval"
    print(f"Running evaluation...")
    !{eval_cmd}
else:
    print(f"No checkpoint found in {OUTPUT_DIR}")
    print("Make sure training has completed.")

In [ ]:
#@title Plot Training Curves (if wandb enabled)
if USE_WANDB:
    import wandb
    api = wandb.Api()
    
    # Get the latest run
    runs = api.runs(f"{WANDB_PROJECT}")
    if runs:
        latest_run = runs[0]
        print(f"Latest run: {latest_run.name}")
        print(f"URL: {latest_run.url}")
    else:
        print("No runs found")
else:
    print("Wandb logging not enabled. Set USE_WANDB=True in configuration.")

## 5. Save to Google Drive

In [ ]:
#@title Copy Checkpoints to Google Drive
import shutil
import os

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/TinyViT_checkpoints"  #@param {type:"string"}

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Copy output directory to drive
if os.path.exists(OUTPUT_DIR):
    dest = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(OUTPUT_DIR))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(OUTPUT_DIR, dest)
    print(f"Copied {OUTPUT_DIR} to {dest}")
else:
    print(f"Output directory {OUTPUT_DIR} not found")

## 6. Quick Experiment Presets

Run these cells to quickly set up common experiments.

In [ ]:
#@title Preset: Finetune TinyViT-21M Teacher on CIFAR-100
# This trains the teacher model for later distillation

EXPERIMENT_TYPE = "finetune_teacher"
MODEL_NAME = "TinyViT-21M-Teacher-CIFAR100"
STUDENT_TYPE = "tiny_vit_21m"
STUDENT_CONFIG = MODEL_CONFIGS[STUDENT_TYPE]
PRETRAINED_PATH = "pretrained/tiny_vit_21m_22k_distill.pth"
OUTPUT_DIR = "./output/teacher_tinyvit21m"
EPOCHS = 50
BATCH_SIZE = 64
BASE_LR = 2e-4

print("Preset loaded: Finetune TinyViT-21M Teacher")
print("Now run 'Generate Config YAML' and 'Run Training' cells")

In [ ]:
#@title Preset: Online Distillation (TinyViT-21M -> TinyViT-5M) with Features
# This trains the student with online logits + feature distillation

EXPERIMENT_TYPE = "online_distill_features"
MODEL_NAME = "TinyViT-5M-OnlineDistill-Features"
STUDENT_TYPE = "tiny_vit_5m"
STUDENT_CONFIG = MODEL_CONFIGS[STUDENT_TYPE]
TEACHER_TYPE = "tiny_vit_21m"
TEACHER_CONFIG = MODEL_CONFIGS[TEACHER_TYPE]
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/TinyViT-21M-Teacher-CIFAR100/default/ckpt_best.pth"
OUTPUT_DIR = "./output/student_online_distill_features"
FEATURE_ENABLED = True
FEATURE_WEIGHT = 0.5
EPOCHS = 200
BATCH_SIZE = 64
BASE_LR = 2e-3

print("Preset loaded: Online Distillation with Features")
print("Make sure you have trained the teacher first!")
print("Now run 'Generate Config YAML' and 'Run Training' cells")

In [ ]:
#@title Preset: Online Distillation (Logits Only) - Baseline
# This trains the student with online logits only (no feature distillation)

EXPERIMENT_TYPE = "online_distill"
MODEL_NAME = "TinyViT-5M-OnlineDistill-LogitsOnly"
STUDENT_TYPE = "tiny_vit_5m"
STUDENT_CONFIG = MODEL_CONFIGS[STUDENT_TYPE]
TEACHER_TYPE = "tiny_vit_21m"
TEACHER_CONFIG = MODEL_CONFIGS[TEACHER_TYPE]
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/TinyViT-21M-Teacher-CIFAR100/default/ckpt_best.pth"
OUTPUT_DIR = "./output/student_online_distill_logits_only"
FEATURE_ENABLED = False
EPOCHS = 200
BATCH_SIZE = 64
BASE_LR = 2e-3

print("Preset loaded: Online Distillation (Logits Only)")
print("Make sure you have trained the teacher first!")
print("Now run 'Generate Config YAML' and 'Run Training' cells")

In [ ]:
#@title Preset: Feature Weight Ablation (beta=0.25)

EXPERIMENT_TYPE = "online_distill_features"
MODEL_NAME = "TinyViT-5M-OnlineDistill-Beta0.25"
STUDENT_TYPE = "tiny_vit_5m"
STUDENT_CONFIG = MODEL_CONFIGS[STUDENT_TYPE]
TEACHER_TYPE = "tiny_vit_21m"
TEACHER_CONFIG = MODEL_CONFIGS[TEACHER_TYPE]
TEACHER_CHECKPOINT = "./output/teacher_tinyvit21m/TinyViT-21M-Teacher-CIFAR100/default/ckpt_best.pth"
OUTPUT_DIR = "./output/student_beta_0.25"
FEATURE_ENABLED = True
FEATURE_WEIGHT = 0.25  # Changed from 0.5
EPOCHS = 200
BATCH_SIZE = 64
BASE_LR = 2e-3

print("Preset loaded: Feature Weight Ablation (beta=0.25)")
print("Now run 'Generate Config YAML' and 'Run Training' cells")